In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================
data_dir = Path('data')

# Fichiers de données disponibles (noms exacts tels que fournis)
FILES_CONFIG = {
    'NO2': data_dir / '2024_NO2_boulevard_périphérique.csv',
    'PM25': data_dir / '2026_PM25_boulevard_périphérique.csv',
    'PM10': data_dir / '2026_PM10_boulevard_périphérique.csv',
}

# Segments du périphérique (colonnes du dataset)
SEGMENTS = [
    'Chap-Bagn', 'Bagn-Berc', 'Berc-Ital', 'Ital-A6a',
    'A6a-Sevr', 'Sevr-Aute', 'Aute-Mail', 'Mail-Chap'
]

# =============================================================================
# FONCTIONS UTILITAIRES
# =============================================================================

def load_pollutant_data(file_path: Path, pollutant_name: str) -> pd.DataFrame:
    """
    Charge un fichier CSV de données de pollution avec gestion des erreurs.
    
    Args:
        file_path: Chemin vers le fichier CSV
        pollutant_name: Nom du polluant (pour le logging)
    
    Returns:
        DataFrame avec les données chargées et pré-traitées
    """
    if not file_path.exists():
        print(f"⚠️  Fichier non trouvé pour {pollutant_name}: {file_path}")
        return pd.DataFrame()
    
    print(f"✅ Chargement de {pollutant_name} : {file_path.name}")
    
    df = pd.read_csv(file_path, sep=",")
    
    # Conversion de la colonne time en datetime
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    
    # Ajout du nom du polluant comme colonne
    df['pollutant'] = pollutant_name
    
    return df

def handle_missing_values(df: pd.DataFrame, segments: list) -> pd.DataFrame:
    """
    Gère les valeurs manquantes dans les colonnes de segments.
    
    Args:
        df: DataFrame à traiter
        segments: Liste des colonnes de segments à vérifier
    
    Returns:
        DataFrame avec statistiques sur les valeurs manquantes
    """
    missing_stats = {}
    
    for segment in segments:
        if segment in df.columns:
            missing_count = df[segment].isna().sum()
            missing_pct = (missing_count / len(df)) * 100 if len(df) > 0 else 0
            missing_stats[segment] = {
                'missing_count': int(missing_count),
                'missing_pct': round(missing_pct, 2)
            }
            
            # Interpolation linéaire pour les valeurs numériques manquantes
            if missing_count > 0 and missing_pct < 50:  # Seulement si <50% de missing
                df[segment] = df[segment].interpolate(method='linear', limit_direction='both')
    
    return df, missing_stats

def compute_statistics(df: pd.DataFrame, segments: list) -> dict:
    """
    Calcule des statistiques descriptives par segment et par polluant.
    
    Returns:
        Dictionnaire avec les statistiques agrégées
    """
    stats = {}
    
    for pollutant in df['pollutant'].unique():
        df_pollutant = df[df['pollutant'] == pollutant]
        stats[pollutant] = {}
        
        for segment in segments:
            if segment in df_pollutant.columns:
                values = df_pollutant[segment].dropna()
                if len(values) > 0:
                    stats[pollutant][segment] = {
                        'mean': round(values.mean(), 2),
                        'median': round(values.median(), 2),
                        'std': round(values.std(), 2),
                        'min': round(values.min(), 2),
                        'max': round(values.max(), 2),
                        'count': int(len(values))
                    }
    
    return stats

def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ajoute des features temporelles pour l'analyse et la modélisation.
    """
    df = df.copy()
    df['hour'] = df['time'].dt.hour
    df['day_of_week'] = df['time'].dt.dayofweek
    df['day_name'] = df['time'].dt.day_name()
    df['month'] = df['time'].dt.month
    df['is_weekend'] = df['day_of_week'] >= 5
    df['is_peak_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19])
    
    return df

# =============================================================================
# FONCTION PRINCIPALE
# =============================================================================

def main():
    """
    Fonction principale : orchestre le chargement et l'analyse des données.
    """
    print("=" * 70)
    print("🌍 OPTIMISATION DE LA MOBILITÉ URBAINE - ANALYSE DES DONNÉES")
    print("=" * 70)
    
    # 1. Chargement des données
    print("\n📥 Étape 1 : Chargement des fichiers de données...")
    all_dfs = []
    
    for pollutant, file_path in FILES_CONFIG.items():
        df = load_pollutant_data(file_path, pollutant)
        if not df.empty:
            all_dfs.append(df)
    
    if not all_dfs:
        print("❌ Aucune donnée chargée. Vérifiez les chemins des fichiers.")
        return
    
    # Concaténation des DataFrames
    df_combined = pd.concat(all_dfs, ignore_index=True)
    print(f"\n✅ Données combinées : {len(df_combined)} lignes, {len(df_combined.columns)} colonnes")
    
    # 2. Gestion des valeurs manquantes
    print("\n🔧 Étape 2 : Gestion des valeurs manquantes...")
    df_cleaned, missing_report = handle_missing_values(df_combined, SEGMENTS)
    
    # Affichage du rapport de missing values
    print("\n📊 Rapport des valeurs manquantes par segment :")
    for pollutant in df_cleaned['pollutant'].unique():
        df_poll = df_cleaned[df_cleaned['pollutant'] == pollutant]
        print(f"\n  {pollutant}:")
        for segment, stats in missing_report.items():
            if stats['missing_count'] > 0:
                print(f"    • {segment}: {stats['missing_count']} manquantes ({stats['missing_pct']}%)")
    
    # 3. Ajout des features temporelles
    print("\n⏰ Étape 3 : Ajout des features temporelles...")
    df_enriched = add_temporal_features(df_cleaned)
    print(f"✅ Features ajoutées : hour, day_of_week, month, is_weekend, is_peak_hour")
    
    # 4. Calcul des statistiques descriptives
    print("\n📈 Étape 4 : Calcul des statistiques descriptives...")
    stats = compute_statistics(df_enriched, SEGMENTS)
    
    # Affichage synthétique des statistiques
    print("\n📊 Statistiques globales par polluant (moyenne sur tous les segments) :")
    for pollutant, seg_stats in stats.items():
        if seg_stats:
            means = [s['mean'] for s in seg_stats.values()]
            print(f"  {pollutant}:")
            print(f"    • Moyenne globale : {np.mean(means):.2f} µg/m³")
            print(f"    • Écart-type moyen : {np.mean([s['std'] for s in seg_stats.values()]):.2f}")
            print(f"    • Plage : {np.min([s['min'] for s in seg_stats.values()]):.2f} - {np.max([s['max'] for s in seg_stats.values()]):.2f}")
    
    # 5. Export des données préparées
    print("\n💾 Étape 5 : Export des données préparées...")
    output_dir = data_dir / 'processed'
    output_dir.mkdir(exist_ok=True)
    
    output_file = output_dir / 'air_quality_peripherique_prepared.csv'
    df_enriched.to_csv(output_file, index=False)
    print(f"✅ Fichier exporté : {output_file}")
    
    # 6. Résumé final
    print("\n" + "=" * 70)
    print("📋 RÉSUMÉ DU TRAITEMENT")
    print("=" * 70)
    print(f"• Période couverte : {df_enriched['time'].min()} → {df_enriched['time'].max()}")
    print(f"• Polluants analysés : {', '.join(df_enriched['pollutant'].unique())}")
    print(f"• Segments du périphérique : {len(SEGMENTS)}")
    print(f"• Nombre total d'observations : {len(df_enriched):,}")
    print(f"• Features créées : 6 (temporelles)")
    print(f"• Données prêtes pour : modélisation, visualisation, dashboard")
    print("=" * 70)
    
    return df_enriched, stats

# =============================================================================
# EXÉCUTION
# =============================================================================

if __name__ == "__main__":
    # Exécuter le pipeline principal
    df_result, statistics = main()
    
    # Bonus : Exemple d'analyse rapide (pic de pollution)
    if df_result is not None and not df_result.empty:
        print("\n🔍 ANALYSE RAPIDE : Identification des pics de pollution")
        
        # Calcul de la moyenne par segment et par heure de pointe
        peak_data = df_enriched[df_enriched['is_peak_hour']].groupby(
            ['pollutant', 'segment' if 'segment' in df_enriched.columns else SEGMENTS[0]]
        ).mean(numeric_only=True)
        
        print("\n💡 Conseil pour le projet :")
        print("   → Utilisez ces données pour entraîner un modèle LSTM/ARIMA")
        print("   → Intégrez les données météo via OpenWeatherMap API")
        print("   → Créez un dashboard Streamlit pour la visualisation")

🌍 OPTIMISATION DE LA MOBILITÉ URBAINE - ANALYSE DES DONNÉES

📥 Étape 1 : Chargement des fichiers de données...
⚠️  Fichier non trouvé pour NO2: data\2026_NO2_boulevard_périphérique.csv
⚠️  Fichier non trouvé pour PM25: data\2026_PM25_boulevard_périphérique.csv
⚠️  Fichier non trouvé pour PM10: data\2026_PM10_boulevard_périphérique.csv
❌ Aucune donnée chargée. Vérifiez les chemins des fichiers.


TypeError: cannot unpack non-iterable NoneType object